# 📈 Time Series Analysis: Ethiopian Tech Salary Trends

## Overview
This notebook analyzes salary trends over time in the Ethiopian tech sector, forecasting future salary patterns and identifying seasonal trends.

### Analysis Components:
- 📊 **Trend Analysis**: Long-term salary growth patterns
- 🔄 **Seasonal Decomposition**: Identifying cyclical patterns
- 📈 **Forecasting Models**: ARIMA, Prophet, and ML-based forecasting
- 🎯 **Department-wise Trends**: Sector-specific growth analysis
- 🌍 **Geographic Trends**: City-wise salary evolution
- 🎓 **Education Impact Over Time**: Changing value of education

---

In [ ]:
# Import comprehensive time series analysis libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Time series specific libraries
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools import adfuller, acf, pacf
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    STATSMODELS_AVAILABLE = True
    print("✅ Statsmodels available for time series analysis")
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("⚠️ Statsmodels not available. Install with: pip install statsmodels")

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print("✅ Prophet available for forecasting")
except ImportError:
    PROPHET_AVAILABLE = False
    print("⚠️ Prophet not available. Install with: pip install prophet")

# Scikit-learn for ML-based forecasting
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Utilities
from IPython.display import display, HTML
from scipy import stats
import joblib

print("\n📈 Time Series Analysis Setup Complete!")
print(f"📊 Statsmodels: {STATSMODELS_AVAILABLE}")
print(f"🔮 Prophet: {PROPHET_AVAILABLE}")
print("🚀 Ready for Comprehensive Time Series Analysis!")

In [ ]:
# Generate synthetic time series data for Ethiopian salary trends
print("📊 GENERATING SYNTHETIC TIME SERIES DATA")
print("=" * 50)

# Load base dataset
df_base = pd.read_csv('../ethiopia_salary_data.csv')
print(f"✅ Base dataset loaded: {df_base.shape[0]} records")

# Generate time series data (2019-2024)
start_date = datetime(2019, 1, 1)
end_date = datetime(2024, 12, 31)
date_range = pd.date_range(start=start_date, end=end_date, freq='M')

print(f"📅 Time range: {start_date.strftime('%Y-%m')} to {end_date.strftime('%Y-%m')}")
print(f"📊 Total months: {len(date_range)}")

# Create synthetic time series data
np.random.seed(42)
time_series_data = []

# Base salary trends with growth and seasonality
base_salary = 600000  # Starting average salary in ETB
annual_growth_rate = 0.08  # 8% annual growth
seasonal_amplitude = 0.05  # 5% seasonal variation

for i, date in enumerate(date_range):
    # Calculate years since start
    years_elapsed = (date - start_date).days / 365.25
    
    # Trend component (exponential growth)
    trend_salary = base_salary * (1 + annual_growth_rate) ** years_elapsed
    
    # Seasonal component (higher salaries in Q4, lower in Q2)
    month = date.month
    seasonal_factor = 1 + seasonal_amplitude * np.sin(2 * np.pi * (month - 3) / 12)
    
    # Random noise
    noise_factor = 1 + np.random.normal(0, 0.02)  # 2% random variation
    
    # Calculate final salary
    monthly_avg_salary = trend_salary * seasonal_factor * noise_factor
    
    # Generate individual records for this month
    n_records = np.random.randint(8, 15)  # 8-15 records per month
    
    for _ in range(n_records):
        # Sample from base dataset for realistic feature combinations
        base_record = df_base.sample(1).iloc[0]
        
        # Adjust salary based on time series trend
        salary_multiplier = monthly_avg_salary / df_base['salary_etb'].mean()
        adjusted_salary = base_record['salary_etb'] * salary_multiplier
        
        # Add some individual variation
        individual_variation = np.random.normal(1, 0.1)
        final_salary = adjusted_salary * individual_variation
        
        time_series_data.append({
            'date': date,
            'year': date.year,
            'month': date.month,
            'quarter': f"Q{(date.month-1)//3 + 1}",
            'employee_id': len(time_series_data) + 1,
            'experience_years': base_record['experience_years'],
            'test_score': base_record['test_score'],
            'department': base_record['department'],
            'education_level': base_record['education_level'],
            'location': base_record['location'],
            'salary_etb': int(final_salary),
            'salary_usd': int(final_salary / 55)
        })

# Create DataFrame
df_ts = pd.DataFrame(time_series_data)
df_ts['date'] = pd.to_datetime(df_ts['date'])
df_ts = df_ts.sort_values('date').reset_index(drop=True)

print(f"\n✅ Time series dataset created: {len(df_ts)} records")
print(f"📊 Date range: {df_ts['date'].min().strftime('%Y-%m')} to {df_ts['date'].max().strftime('%Y-%m')}")
print(f"💰 Salary range: {df_ts['salary_etb'].min():,.0f} - {df_ts['salary_etb'].max():,.0f} ETB")

# Display sample data
print("\n📋 Sample Time Series Data:")
display(df_ts[['date', 'department', 'experience_years', 'salary_etb', 'salary_usd']].head(10))

# Save the time series dataset
df_ts.to_csv('../ethiopia_salary_timeseries.csv', index=False)
print("\n💾 Time series dataset saved as 'ethiopia_salary_timeseries.csv'")